# Детекция фейковых новостей с GigaChat API

В этом ноутбуке реализована детекция фейковых новостей с помощью **GigaChat API** (Сбер) в режиме zero-shot.

Для каждой новости модель возвращает два поля:
- `is_fake` — `true` / `false`  
- `explanation` — краткое объяснение решения

Результаты сохраняются в `models/gigachat/` и сравниваются с другими подходами.

## 1. Установка зависимостей

In [ ]:
!pip install requests scikit-learn pandas matplotlib seaborn tqdm -q

## 2. Импорты и конфигурация

In [ ]:
import os
import json
import time
import uuid
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

# Воспроизводимость
SEED = 42
np.random.seed(SEED)

In [ ]:
@dataclass
class Config:
    # Данные
    data_path:      str = '../../data/ready_dataset.csv'
    text_column:    str = 'combined_text'
    label_column:   str = 'label'
    test_size:      float = 0.2
    sample_size:    int = 100       # сколько примеров брать для оценки (None = все)

    # GigaChat
    # Получить credentials: https://developers.sber.ru/portal/products/gigachat
    # Вставьте сюда ваш Authorization токен (Base64(client_id:client_secret))
    gigachat_credentials: str = 'YOUR_GIGACHAT_CREDENTIALS_HERE'
    gigachat_scope:       str = 'GIGACHAT_API_PERS'  # или GIGACHAT_API_CORP
    gigachat_model:       str = 'GigaChat'
    max_tokens:           int = 256
    temperature:          float = 0.1
    request_delay:        float = 0.5   # пауза между запросами (сек)

    # Сохранение
    output_dir: str = '../../models/gigachat'

cfg = Config()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
print('Config готов. Output dir:', cfg.output_dir)

## 3. Загрузка данных

In [ ]:
df = pd.read_csv(cfg.data_path)
df = df[[cfg.text_column, cfg.label_column]].dropna()
df[cfg.text_column] = df[cfg.text_column].astype(str).str.strip()
df = df[df[cfg.text_column] != '']
df[cfg.label_column] = pd.to_numeric(df[cfg.label_column], errors='coerce')
df = df.dropna(subset=[cfg.label_column])
df[cfg.label_column] = df[cfg.label_column].astype(int)

print(f'Всего записей: {len(df)}')
print(f'Распределение меток:\n{df[cfg.label_column].value_counts().rename({0: "Фейк (0)", 1: "Правда (1)"})}')

In [ ]:
_, test_df = train_test_split(
    df,
    test_size=cfg.test_size,
    random_state=SEED,
    stratify=df[cfg.label_column]
)

if cfg.sample_size and cfg.sample_size < len(test_df):
    test_df = test_df.sample(cfg.sample_size, random_state=SEED)

test_df = test_df.reset_index(drop=True)
print(f'Тестовая выборка: {len(test_df)} примеров')
print(f'Фейк: {(test_df[cfg.label_column] == 0).sum()}, Правда: {(test_df[cfg.label_column] == 1).sum()}')

## 4. GigaChat API клиент

In [ ]:
class GigaChatClient:
    AUTH_URL = 'https://ngw.devices.sberbank.ru:9443/api/v2/oauth'
    API_URL  = 'https://gigachat.devices.sberbank.ru/api/v1/chat/completions'

    def __init__(self, credentials: str, scope: str, model: str,
                 max_tokens: int = 256, temperature: float = 0.1):
        self.credentials = credentials
        self.scope       = scope
        self.model       = model
        self.max_tokens  = max_tokens
        self.temperature = temperature
        self._token      = None
        self._token_exp  = 0

    def _refresh_token(self):
        headers = {
            'Content-Type':  'application/x-www-form-urlencoded',
            'Accept':        'application/json',
            'RqUID':         str(uuid.uuid4()),
            'Authorization': f'Basic {self.credentials}',
        }
        resp = requests.post(
            self.AUTH_URL,
            data={'scope': self.scope},
            headers=headers,
            verify=False,
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        self._token     = data['access_token']
        self._token_exp = data['expires_at'] / 1000  # ms → s
        print('Токен получен, истекает через ~30 мин')

    def _ensure_token(self):
        if self._token is None or time.time() >= self._token_exp - 60:
            self._refresh_token()

    def classify(self, text: str) -> dict:
        """
        Возвращает {'is_fake': bool, 'explanation': str}.
        При ошибке парсинга возвращает {'is_fake': None, 'explanation': str}.
        """
        self._ensure_token()

        system_prompt = (
            'Ты — эксперт по верификации новостей. '
            'Тебе дают текст новости на русском языке. '
            'Определи, является ли новость фейковой. '
            'Ответ верни строго в формате JSON — без markdown, без пояснений вне JSON:\n'
            '{"is_fake": true или false, "explanation": "краткое объяснение (1-3 предложения)"}'
        )

        user_prompt = f'Текст новости:\n{text[:1500]}'

        payload = {
            'model': self.model,
            'messages': [
                {'role': 'system',  'content': system_prompt},
                {'role': 'user',    'content': user_prompt},
            ],
            'max_tokens':  self.max_tokens,
            'temperature': self.temperature,
            'stream':      False,
        }

        headers = {
            'Authorization': f'Bearer {self._token}',
            'Content-Type':  'application/json',
            'Accept':        'application/json',
        }

        resp = requests.post(
            self.API_URL,
            json=payload,
            headers=headers,
            verify=False,
            timeout=60,
        )
        resp.raise_for_status()
        content = resp.json()['choices'][0]['message']['content'].strip()

        return self._parse_response(content)

    @staticmethod
    def _parse_response(content: str) -> dict:
        # Убираем ```json ... ``` если модель всё же добавила
        content = content.strip()
        if content.startswith('```'):
            content = content.split('```')[1]
            if content.startswith('json'):
                content = content[4:]
        content = content.strip()

        try:
            data = json.loads(content)
            is_fake = bool(data['is_fake'])
            explanation = str(data.get('explanation', '')).strip()
            return {'is_fake': is_fake, 'explanation': explanation}
        except Exception:
            # Fallback: попробуем найти true/false в тексте
            lower = content.lower()
            if '"is_fake": true' in lower or '"is_fake":true' in lower:
                return {'is_fake': True,  'explanation': content}
            if '"is_fake": false' in lower or '"is_fake":false' in lower:
                return {'is_fake': False, 'explanation': content}
            return {'is_fake': None, 'explanation': content}


client = GigaChatClient(
    credentials=cfg.gigachat_credentials,
    scope=cfg.gigachat_scope,
    model=cfg.gigachat_model,
    max_tokens=cfg.max_tokens,
    temperature=cfg.temperature,
)
print('Клиент создан')

## 5. Проверка одного запроса

In [ ]:
sample_text = test_df[cfg.text_column].iloc[0]
sample_label = test_df[cfg.label_column].iloc[0]

print(f'Текст (первые 300 символов):\n{sample_text[:300]}\n')
print(f'Истинная метка: {"Фейк" if sample_label == 0 else "Правда"} ({sample_label})')
print('\nЗапрос к GigaChat...')

result = client.classify(sample_text)
print(f'\nОтвет модели:')
print(f'  is_fake:     {result["is_fake"]}')
print(f'  explanation: {result["explanation"]}')

## 6. Классификация тестовой выборки

In [ ]:
results = []
errors  = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc='GigaChat classify'):
    text  = row[cfg.text_column]
    label = row[cfg.label_column]

    try:
        prediction = client.classify(text)
        results.append({
            'idx':         idx,
            'true_label':  label,
            'is_fake':     prediction['is_fake'],
            'explanation': prediction['explanation'],
            'error':       None,
        })
    except Exception as e:
        error_msg = str(e)
        errors.append({'idx': idx, 'error': error_msg})
        results.append({
            'idx':         idx,
            'true_label':  label,
            'is_fake':     None,
            'explanation': '',
            'error':       error_msg,
        })

    time.sleep(cfg.request_delay)

results_df = pd.DataFrame(results)
print(f'\nГотово. Всего: {len(results_df)}, ошибок: {len(errors)}, None-ответов: {results_df["is_fake"].isna().sum()}')

In [ ]:
# Сохраняем сырые результаты
results_path = Path(cfg.output_dir) / 'predictions.csv'
results_df.to_csv(results_path, index=False, encoding='utf-8')
print(f'Сохранено: {results_path}')

# Показываем несколько примеров
results_df[['true_label', 'is_fake', 'explanation']].head(5)

## 7. Оценка качества

In [ ]:
# Оставляем только строки с валидным ответом модели
valid_df = results_df.dropna(subset=['is_fake']).copy()
print(f'Валидных ответов: {len(valid_df)} из {len(results_df)}')

# GigaChat: is_fake=True → label=0 (фейк), is_fake=False → label=1 (правда)
y_pred = (~valid_df['is_fake'].astype(bool)).astype(int).values
y_true = valid_df['true_label'].values

acc  = accuracy_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred, average='weighted')
prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)

try:
    auc = roc_auc_score(y_true, y_pred)
except Exception:
    auc = float('nan')

metrics = {
    'accuracy':  round(acc,  4),
    'f1':        round(f1,   4),
    'precision': round(prec, 4),
    'recall':    round(rec,  4),
    'roc_auc':   round(auc,  4),
    'n_valid':   int(len(valid_df)),
    'n_total':   int(len(results_df)),
}

for k, v in metrics.items():
    print(f'  {k:12s}: {v}')

In [ ]:
# Сохраняем метрики
metrics_path = Path(cfg.output_dir) / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print(f'Метрики сохранены: {metrics_path}')

## 8. Визуализация

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('GigaChat — детекция фейковых новостей', fontsize=14, fontweight='bold')

# --- Confusion matrix ---
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=['Фейк (0)', 'Правда (1)'],
    yticklabels=['Фейк (0)', 'Правда (1)'],
)
axes[0].set_title('Матрица ошибок')
axes[0].set_ylabel('Истинная метка')
axes[0].set_xlabel('Предсказание')

# --- Метрики ---
metric_names  = ['Accuracy', 'F1', 'Precision', 'Recall']
metric_values = [acc, f1, prec, rec]
colors = ['#4A9EF5', '#2ECC71', '#F39C12', '#E74C3C']

bars = axes[1].bar(metric_names, metric_values, color=colors, width=0.5)
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Метрики качества')
axes[1].set_ylabel('Значение')

for bar, val in zip(bars, metric_values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f'{val:.3f}', ha='center', va='bottom', fontweight='bold'
    )

plt.tight_layout()
plot_path = Path(cfg.output_dir) / 'metrics_plot.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'График сохранён: {plot_path}')

## 9. Примеры предсказаний

In [ ]:
def label_name(val):
    return 'Правда' if val == 1 else 'Фейк'

# Корректные предсказания
correct = valid_df[y_true == y_pred].sample(min(3, (y_true == y_pred).sum()), random_state=SEED)
print('=== Верные предсказания ===')
for _, row in correct.iterrows():
    print(f'  Метка: {label_name(int(row["true_label"]))}')
    print(f'  Объяснение: {row["explanation"][:200]}')
    print()

# Ошибки
wrong = valid_df[y_true != y_pred]
if len(wrong):
    wrong_sample = wrong.sample(min(3, len(wrong)), random_state=SEED)
    print('=== Ошибки ===')
    for _, row in wrong_sample.iterrows():
        pred_label = 1 - int(row['is_fake'])  # is_fake=True → pred=0
        print(f'  Истинная: {label_name(int(row["true_label"]))}, Предсказана: {label_name(pred_label)}')
        print(f'  Объяснение: {row["explanation"][:200]}')
        print()

## 10. Classification report

In [ ]:
print(classification_report(
    y_true, y_pred,
    target_names=['Фейк (0)', 'Правда (1)'],
    digits=4
))

## 11. Итог

| Метрика    | Значение |
|------------|----------|
| Accuracy   | см. выше |
| F1         | см. выше |
| Precision  | см. выше |
| Recall     | см. выше |

**Выводы:**
- GigaChat в режиме zero-shot классифицирует новости без какого-либо обучения на целевом датасете.
- Модель возвращает структурированный JSON с полями `is_fake` и `explanation`.
- Основное ограничение — стоимость и задержка API; для масштабного использования предпочтительны fine-tuned модели.
- Преимущество — объяснение решения на естественном языке.